In [ ]:
# Processa todas as imagens (descomente para executar)
# for img_file in images:
#     img_path = os.path.join(image_dir, img_file)
#     print(f"\n{'='*60}")
#     try:
#         plot_comparison(img_path, config, figsize=(14, 8))
#     except Exception as e:
#         print(f"❌ Erro ao processar {img_file}: {e}")

print("\n✓ Script de demonstração pronto!")

## 6. Processamento em Lote (Opcional)

Descomente a célula abaixo para processar todas as imagens do diretório.

In [ ]:
# Cria diretório de resultados se não existir
os.makedirs('./resultados', exist_ok=True)

# Salva os resultados
base_name = os.path.splitext(images[0])[0]
path_classic = f'./resultados/{base_name}_canny_classic.png'
path_mod = f'./resultados/{base_name}_canny_modificado.png'

save_image(edges_classic, path_classic)
save_image(edges_mod, path_mod)

print(f"\n✓ Resultados salvos em ./resultados/")

## 5. Salvando Resultados

In [ ]:
# Seleciona primeira imagem disponível
if images:
    test_image = os.path.join(image_dir, images[0])
    img_original, edges_classic, edges_mod = plot_comparison(test_image, config)

## 4. Testando com Primeira Imagem Disponível

Selecione uma imagem para análise detalhada. O Canny Clássico pode perder bordas cromáticas (cores sem contraste em cinza), enquanto o Canny Modificado as preserva.

In [ ]:
def plot_comparison(image_path, config, figsize=(16, 10)):
    """
    Função para processar uma imagem e exibir comparação lado a lado.
    
    Args:
        image_path: Caminho da imagem
        config: Dicionário com configurações
        figsize: Tamanho da figura
    """
    print(f"\n🔍 Processando: {image_path}")
    print("-" * 60)
    
    # Carrega a imagem
    image = read_image(image_path)
    print(f"✓ Imagem carregada: {image.shape}")
    
    # Canny Clássico
    print("\n⏳ Executando Canny Clássico...")
    canny_classic = CannyClassic(
        sigma=config.get('sigma', 1.0),
        kernel_size=config.get('kernel_size', 5),
        low_threshold=config.get('low_threshold', 100),
        high_threshold=config.get('high_threshold', 200)
    )
    edges_classic = canny_classic.detect(image)
    print("✓ Canny Clássico concluído")
    
    # Canny Modificado
    print("\n⏳ Executando Canny Modificado (Gabor-Di Zenzo)...")
    canny_mod = CannyModificado(
        sigma_blur=config.get('sigma', 1.0),
        num_gabor_orientations=config.get('gabor_orientations', 8),
        num_gabor_frequencies=config.get('gabor_frequencies', 3),
        low_threshold=config.get('low_threshold', 100),
        high_threshold=config.get('high_threshold', 200)
    )
    edges_mod = canny_mod.detect(image)
    print("✓ Canny Modificado concluído")
    
    # Visualização
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)
    
    # Original
    ax1 = fig.add_subplot(gs[0, :])
    ax1.imshow(image)
    ax1.set_title('Imagem Original (RGB)', fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # Canny Clássico
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.imshow(edges_classic, cmap='gray')
    ax2.set_title('Canny Clássico', fontsize=12, fontweight='bold', color='blue')
    ax2.axis('off')
    
    # Canny Modificado
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.imshow(edges_mod, cmap='gray')
    ax3.set_title('Canny Modificado (Gabor-Di Zenzo)', fontsize=12, fontweight='bold', color='green')
    ax3.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Comparação visual exibida")
    
    # Estatísticas
    n_edges_classic = np.count_nonzero(edges_classic > 128)
    n_edges_mod = np.count_nonzero(edges_mod > 128)
    
    print(f"\n📊 Estatísticas:")
    print(f"  - Pixels de borda (Clássico): {n_edges_classic}")
    print(f"  - Pixels de borda (Modificado): {n_edges_mod}")
    print(f"  - Diferença: {abs(n_edges_classic - n_edges_mod)}")
    
    return image, edges_classic, edges_mod

print("✓ Função de comparação definida")

## 3. Função Auxiliar para Comparação

In [ ]:
import json

# Carrega configurações
config_path = './config/params.json'
with open(config_path, 'r') as f:
    config = json.load(f)

print("⚙️ Configurações Carregadas:")
print(f"  - Sigma (desfoque): {config['sigma']}")
print(f"  - Kernel size: {config['kernel_size']}")
print(f"  - Limiar baixo: {config['low_threshold']}")
print(f"  - Limiar alto: {config['high_threshold']}")
print(f"  - Orientações Gabor: {config['gabor_orientations']}")
print(f"  - Frequências Gabor: {config['gabor_frequencies']}")

## 2. Carregando Configurações

In [ ]:
# Verifica imagens disponíveis
image_dir = './imagens'
if os.path.exists(image_dir):
    images = [f for f in os.listdir(image_dir) 
              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))]
    print(f"📁 Imagens encontradas ({len(images)}):")
    for i, img in enumerate(images, 1):
        print(f"   {i}. {img}")
else:
    print(f"❌ Diretório de imagens não encontrado: {image_dir}")

## 1. Verificar Imagens Disponíveis

In [ ]:
import sys
sys.path.insert(0, './src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os
from PIL import Image

# Importa os módulos locais
from utils import read_image, save_image
from canny_classic import CannyClassic
from canny_modificado import CannyModificado

print("✓ Bibliotecas importadas com sucesso!")

# Detecção de Bordas: Canny Clássico vs. Modificado (Gabor-Di Zenzo)

Este notebook demonstra a comparação entre o algoritmo Canny tradicional (escalar) e a abordagem modificada com Bancos de Filtros de Gabor e processamento vetorial de Di Zenzo.

## Objetivos
- Implementar e testar o Canny Clássico
- Implementar o Canny Modificado com Gabor + Di Zenzo
- Comparar resultados em diferentes imagens
- Visualizar a preservação de bordas cromáticas